# 02 — Human–Machine Theme Alignment

This notebook aligns manually coded human themes with machine topics via cosine similarity
and reports Gate A alignment metrics.

**Inputs:**  
- `data/human_sentences.csv`  
- `data/theme_list.csv`  
- `Outputs/topic_assignment.csv`  
- `Outputs/topic_centroids.csv`

**Outputs:**  
- `Outputs/themes_x_ALLtopics_cosine.csv` 
- `Outputs/themes_x_TOP5topics_cosine.csv`
- `Outputs/best_matches.csv`  
- `Outputs/gateA_summary.txt`


In [ ]:
# --- Reproducibility Settings ---
import random, numpy as np, torch
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import os
import re
import pickle

In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Load
sentences  = pd.read_csv("../data/human_sentences.csv")
chunks = pd.read_csv("../Outputs/topic_assignment.csv")
semantic_chunks = pd.read_csv("../Outputs/semantic_chunks_docids.csv")
top_words = pd.read_csv('../Outputs/topic_top_words.csv')
themes = pd.read_csv('../data/theme_list.csv')

chunks = pd.concat([chunks, semantic_chunks[['speaker', 'section']]], axis =1)


In [ ]:
#Data Preprocessing

#Filter themes 'include_ge_30==True'
themes = themes[themes["include_ge_30"] == True]


#We need to create columns to join sentences and assignments (chunks)
cat = {'GenAI in nursing education':'NE', 'GenAI in nursing practice':'NP',
       'GenAI demonstration':'EV'}

sentences['section'] = sentences.lvl1.map(cat)

sentences["speaker"] = (
    sentences["doc_id"]
    .str.replace(r"^Validated\s*", "", regex=True)   # drop "Validated " at start
    .str.replace(r"\s*transcript$", "", regex=True) # drop " transcript" at end
)

# #Reload EV, NE, and NP
# ev_csv = pd.read_csv("Processed data (All)/processed_EV.csv")
# ne_csv = pd.read_csv("Processed data (All)/processed_NE.csv")
# np_csv = pd.read_csv("Processed data (All)/processed_NP.csv")

# ev_csv["section"] = "EV"
# ne_csv["section"] = "NE"
# np_csv["section"] = "NP"

# #concat EV, NE, and NP on rows
# all_df = pd.concat([ev_csv, ne_csv, np_csv], ignore_index=True)

# #concat assignments with EV, NE, and NP on column
# merged_df = pd.concat([all_df[['speaker', 'section']], assignments], axis=1 )

In [ ]:
# # Keep only relevant themes

human = sentences.merge(
    themes[["lvl1", "lvl2"]],
    on=["lvl1", "lvl2"],
    how="inner"
)

print(len(sentences))
print(len(human))

In [ ]:
def tokenize(text):
    """Lowercase, remove punctuation, filter stopwords, keep ≥2 letters."""
    cleaned = re.sub(r"[^A-Za-z\s]", "", str(text))  # remove punctuation, keep spaces
    words = re.findall(r"[A-Za-z]+", cleaned.lower())
    #return [w for w in words if len(w) >= 2 and w not in ENGLISH_STOP_WORDS]
    #return [w for w in words if w not in ENGLISH_STOP_WORDS]
    return words

# Standardize ChatGPT spellings
for col in ["sentence_text"]:
    human[col] = human[col].str.replace(r"(?i)\b\w*chat\s*-*\s*gpt\w*\b", "ChatGPT", regex=True)

for col in ["sentence"]:
    chunks[col] = chunks[col].str.replace(r"(?i)\b\w*chat\s*-*\s*gpt\w*\b", "ChatGPT", regex=True)

#  Pre-tokenize both DataFrames once 
human["tokens"]  = human["sentence_text"].astype(str).apply(tokenize)
chunks["tokens"] = chunks["sentence"].astype(str).apply(tokenize)

aligned = []
OVERLAP_THR = 0.75

# Loop through each (speaker, section) 
for (spk, sec), h_sub in human.groupby(["speaker", "section"]):
    c_sub = chunks[(chunks["speaker"] == spk) & (chunks["section"] == sec)]
    if c_sub.empty:
        continue
        

    # Convert chunk tokens to sets for faster overlap computation
    chunk_sets = [set(toks) for toks in c_sub["tokens"]]

    #  Compare each sentence
    for _, row in h_sub.iterrows():
        # Skip sentences with fewer than 3 words
        if len(str(row["sentence_text"]).split()) < 3:
            continue
        sent_set = set(row["tokens"])
        if not sent_set:
            base = row.to_dict()
            base["human_sentence"] = base.pop("sentence_text")   #  rename field
            aligned.append({
                **base,
                "chunk_doc_id": np.nan,
                "chunk_sentence": None,                           
                "topic_id": np.nan,
                "topic_prob": np.nan,
                "overlap": 0.0
            })
            continue

        # Compute overlap ratio with every chunk
        overlaps = [
            len(sent_set & chunk_set) / len(sent_set)     #& is set intersection
            for chunk_set in chunk_sets
        ]

        best_idx  = int(np.argmax(overlaps))
        best_ovlp = overlaps[best_idx]

        base = row.to_dict()
        base["human_sentence"] = base.pop("sentence_text")  #  rename field

        if best_ovlp >= OVERLAP_THR:   # overlap threshold
            best = c_sub.iloc[best_idx]
            aligned.append({
                **base,
                "chunk_doc_id": best["doc_id"],
                "chunk_sentence": best["sentence"],              
                "topic_id": best["topic_id"],
                "topic_prob": best.get("prob", np.nan),
                "overlap": best_ovlp
            })
        else:
            aligned.append({
                **base,
                "chunk_doc_id": np.nan,
                "chunk_sentence": None,                        
                "topic_id": np.nan,
                "topic_prob": np.nan,
                "overlap": best_ovlp
            })

aligned_df = pd.DataFrame(aligned)
aligned_df["topic_id"] = aligned_df["topic_id"].astype("Int64")
aligned_df["chunk_doc_id"] = aligned_df["chunk_doc_id"].astype("Int64")

# aligned_df.to_csv("human_sentences_aligned_by_overlap.csv", index=False, na_rep="NA")

print(
    f"✅ Aligned {aligned_df['topic_id'].notna().sum()} / {len(aligned_df)} sentences "
    f"({100 * aligned_df['topic_id'].notna().mean():.1f}% ≥ {OVERLAP_THR:.2f} overlap)."
)


In [ ]:
#sentences with fewer than 3 words

skipped_short = sum(len(str(s).split()) < 3 for s in human["sentence_text"])
print(f"Skipped short (<3-word) sentences: {skipped_short}")
print(f"Expected remaining: {len(human) - skipped_short}")


In [ ]:
# Load model 
model = SentenceTransformer("all-MiniLM-L6-v2")

df = aligned_df.copy()
#   Define cache path 
cache_file = "sentence_embeddings_chunks.pkl"

if os.path.exists(cache_file):
    print("Loading cached embeddings...")
    with open(cache_file, "rb") as f:
        embeddings = pickle.load(f)
else:
    print("Encoding sentences...")
    embeddings = model.encode(
        df["human_sentence"].tolist(),
        convert_to_numpy=True,
        show_progress_bar=True
    )
    with open(cache_file, "wb") as f:
        pickle.dump(embeddings, f)

df["embedding"] = list(embeddings)

In [ ]:
#  Compute theme centroids 
# Make a unique key for themes
df["theme_key"] = df["lvl1"] + " > " + df["lvl2"]

theme_centroids = (
    df.groupby("theme_key")["embedding"]
    .apply(lambda x: np.mean(np.vstack(x), axis=0))
    
    #For each group (x = a Series of embeddings for that theme):
    #np.vstack(x) stacks all the sentence vectors into a 2D matrix, shape (n_sentences, 384).
    #np.mean(..., axis=0) averages across the rows → gives one centroid vector of length 384.    
)

print("Number of themes:", len(theme_centroids))
print("Vector size:", theme_centroids.iloc[0].shape)


#  Load topic centroids
#Note that outliers had been removed.

topic_centroids = pd.read_csv("../Outputs/topic_centroids.csv")


print("Number of topics:", len(topic_centroids))
print("Vector size:", topic_centroids.drop(columns=["topic_id"]).iloc[0].shape)

In [ ]:
#  Build Cosine Matrix  

theme_keys = theme_centroids.index.tolist()
theme_vecs = np.vstack(theme_centroids.values)   # numeric (384)

# Drop the ID column in topic_centroid before stacking
topic_ids = topic_centroids["topic_id"].tolist()
topic_vecs = topic_centroids.drop(columns=["topic_id"]).to_numpy(dtype=float)

# Double-check dimensions
print("Theme vecs:", theme_vecs.shape)
print("Topic vecs:", topic_vecs.shape)

# Compute cosine matrix
cos_matrix = cosine_similarity(theme_vecs, topic_vecs)
cos_df = pd.DataFrame(cos_matrix, index=theme_keys, columns=topic_ids)

# Save cosine matrix
cos_df.to_csv("../Outputs/themes_x_ALLtopics_cosine.csv",  index=True)
print("✅ Wrote Outputs/themes_x_ALLtopics_cosine.csv")



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#N =  len(cos_df.columns) 
#cos_subset = cos_df[cos_df.columns[:N]]

plt.figure(figsize=(12, 6))
sns.heatmap(
    cos_df,
    cmap="YlGnBu",           # blue-yellow gradient
    annot=True,              # show cosine values in cells
    fmt=".2f",               # two decimals
    linewidths=0.5,
    cbar_kws={"label": "Cosine Similarity"}
)

plt.title(f"Model 2 Theme–Topic Cosine Similarity Heatmap \n\n", size = 12)
plt.xlabel("Topic ID", size = 12)
plt.ylabel("Human Themes", size = 12)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
#Topic sizes
topic_sizes = (
    chunks[chunks["topic_id"] != -1]
    .groupby("topic_id")
    .size()
    .sort_values(ascending=False)
)

#TOP-5 topics matrix by size
top5_ids = topic_sizes.head(5).index.tolist()
cos_df[top5_ids].to_csv("../Outputs/themes_x_TOP5topics_cosine.csv", index=True)
print("✅ Wrote Outputs/themes_x_TOP5topics_cosine.csv")


In [ ]:
#Build best-matches table
best_matches = []
for theme in cos_df.index:
    row = cos_df.loc[theme]
    best_topic = row.idxmax()
    best_score = row.max()
    size = topic_sizes.get(best_topic, 0)

    terms = (
        top_words[top_words["topic_id"] == best_topic]["top_words"].tolist()
    )

    best_matches.append({
        "theme": theme,
        "best_topic": best_topic,
        "cosine": best_score,
        "topic_size": size,
        "top_terms": ", ".join(terms)
    })

best_matches_df = pd.DataFrame(best_matches)
best_matches_df.to_csv("../Outputs/best_matches.csv", index=False)
print("✅ Wrote Outputs/best_matches.csv")

In [ ]:
#Gate A Summary 
tau = 0.4
match_rate = (best_matches_df["cosine"] >= tau).mean()

summary_line = (
    f"Top-1 Match Rate @ τ={tau}: {match_rate:.2f}\n"
    "This means that {:.0f}% of human themes had a best-matching machine topic "
    "whose cosine similarity was at least {:.2f}, indicating strong alignment.\n\n"
).format(match_rate * 100, tau)

with open("../Outputs/gateA_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_line)
    f.write("Bullets:\n")
    f.write("- Themes well-aligned: {}\n\n".format(
        best_matches_df[best_matches_df["cosine"] >= tau]["theme"].tolist()
    ))
    f.write("- Themes poorly aligned: {}\n".format(
        best_matches_df[best_matches_df["cosine"] < tau]["theme"].tolist()
    ))

    

print("✅ Wrote Outputs/gateA_summary.txt")